# Sync exam study guides
Attach the `ExamCatalog` lakehouse as the default, then select **Run all**. Run it whenever you want a refresh, or schedule it to run daily. To support another exam, add its code to `EXAMS` in the last cell.

In [ ]:
"""
Parse a Microsoft Learn exam study guide into Exam / Domain / SubCategory / Skill rows.

Python port of the Power Query in MicrosoftExamSkillsMeasured.xlsx, for use in a
Fabric notebook. Same idea: take the HTML between id="skills-at-a-glance" and
id="study-resources", then walk h3 (domain) -> h4 (subcategory) -> li (skill). Reading stops at the
first h2 after skills are found, so only the current skills list is read when the page also shows the prior one.

Differences from the M version:
  * Exam code comes from the URL / argument, so it works for AZ-, PL-, AI-, etc.,
    not only "Exam DP-".
  * Domain weight "(25–30%)" is split into weight_min / weight_max columns.
  * Each skill gets a stable `skill_key` (hash of exam + domain + subcategory + text)
    so user progress isn't tied to row position. Row numbers shift when Microsoft
    inserts or removes a skill; the key only changes if that skill's wording changes.
  * Fails loudly if the page structure changes, so a bad refresh can't wipe the table.

Usage in a Fabric notebook:
    rows = parse_study_guide(fetch_study_guide("dp-600"), "DP-600")
    spark.createDataFrame(rows).write.mode("overwrite").saveAsTable("exam_skills")
"""
from __future__ import annotations

import hashlib
import re
from html.parser import HTMLParser

import requests

BASE_URL = "https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/{code}"
WEIGHT_RE = re.compile(r"\((\d+)\s*[–-]\s*(\d+)\s*%\)\s*$")


def fetch_study_guide(exam_code: str) -> str:
    url = BASE_URL.format(code=exam_code.lower())
    resp = requests.get(url, timeout=30, headers={"User-Agent": "exam-study-tracker"})
    resp.raise_for_status()
    # Learn pages are UTF-8 but don't always say so in the Content-Type header, and
    # requests then falls back to ISO-8859-1, turning "10–15%" into "10â€“15%".
    return resp.content.decode("utf-8")


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def _skill_key(*parts: str) -> str:
    norm = "|".join(_clean(p).lower() for p in parts)
    return hashlib.sha1(norm.encode("utf-8")).hexdigest()[:16]


class _GuideHTML(HTMLParser):
    """Collects (tag, text) for h1 anywhere, and h3/h4/li between the two section anchors.
    Standard library only, so it runs on the Fabric Spark runtime without installs."""
    CAPTURE = {"h1", "h2", "h3", "h4", "li"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.state = "before"          # before -> inside -> after
        self.items: list[tuple[str, str]] = []
        self.h1 = None
        self._tag, self._buf, self._depth = None, [], 0

    def handle_starttag(self, tag, attrs):
        el_id = dict(attrs).get("id")
        if el_id == "skills-at-a-glance" and self.state == "before":
            self.state = "inside"
            return                      # the anchor heading itself is not a domain
        if self.state == "inside" and el_id == "study-resources":
            self._flush()
            self.state = "after"
        if self._tag is None and tag in self.CAPTURE and (self.state == "inside" or tag == "h1"):
            self._tag, self._buf, self._depth = tag, [], 1
        elif self._tag == tag:
            self._depth += 1

    def handle_endtag(self, tag):
        if self._tag == tag:
            self._depth -= 1
            if self._depth == 0:
                self._flush()

    def handle_data(self, data):
        if self._tag:
            self._buf.append(data)

    def _flush(self):
        if self._tag:
            text = _clean(" ".join(self._buf))
            if self._tag == "h1":
                self.h1 = self.h1 or text
            elif self.state == "inside" and text:
                self.items.append((self._tag, text))
        self._tag, self._buf, self._depth = None, [], 0


def _scan(html: str) -> _GuideHTML:
    p = _GuideHTML()
    p.feed(html)
    p.close()
    return p


def parse_exam_title(html: str, exam_code: str) -> str:
    """Page H1, e.g. 'Study guide for Exam DP-600: Implementing Analytics Solutions Using Microsoft Fabric'."""
    title = _scan(html).h1 or exam_code.upper()
    m = re.search(rf"{re.escape(exam_code.upper())}\s*:\s*(.+)$", title, re.I)
    return m.group(1).strip() if m else title


def parse_study_guide(html: str, exam_code: str, min_skills: int = 10) -> list[dict]:
    exam_code = exam_code.upper()
    scan = _scan(html)
    if scan.state != "after":
        raise ValueError(f"{exam_code}: section anchors not found; page layout may have changed")

    rows: list[dict] = []
    domain = subcat = None
    domain_ids: dict[str, int] = {}
    subcat_ids: dict[tuple[str, str], int] = {}
    seen: set[str] = set()

    for tag, text in scan.items:
        if tag == "h2":
            # During an exam update the page carries a second "skills measured prior to
            # <date>" list under its own h2. Keep only the first (current) list.
            if rows:
                break
            domain = subcat = None
        elif tag == "h3":
            domain, subcat = text, None
            domain_ids.setdefault(domain, len(domain_ids) + 1)
        elif tag == "h4" and domain:
            subcat = text
            subcat_ids.setdefault((domain, subcat), len(subcat_ids) + 1)
        elif tag == "li" and domain and subcat and len(text) > 2:
            # Key on the domain name without its weight, so Microsoft re-weighting a
            # domain (e.g. 25–30% -> 20–25%) doesn't orphan anyone's progress.
            key = _skill_key(exam_code, WEIGHT_RE.sub("", domain).strip(), subcat, text)
            if key in seen:
                continue
            seen.add(key)
            m = WEIGHT_RE.search(domain)
            rows.append({
                "exam": exam_code,
                "domain_number": domain_ids[domain],
                "domain": domain,
                "domain_name": WEIGHT_RE.sub("", domain).strip(),
                "weight_min": int(m.group(1)) if m else None,
                "weight_max": int(m.group(2)) if m else None,
                "subcategory_number": subcat_ids[(domain, subcat)],
                "subcategory": subcat,
                "skill_number": len(rows) + 1,
                "skill": text,
                "skill_key": key,
            })

    if len(rows) < min_skills:
        raise ValueError(f"{exam_code}: only {len(rows)} skills parsed; refusing to overwrite")
    return rows


In [ ]:
# Sync cell: runs after the parser cell in the Fabric notebook (default lakehouse attached).
# Writes two Delta tables that the app reads through a read-only lakehouse connector:
#   exams        one row per exam in EXAMS
#   exam_skills  every skill ever seen; is_current = False when Microsoft removes it,
#                so a user's progress on a retired skill is kept, not orphaned.
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (BooleanType, LongType, StringType, StructField,
                               StructType, TimestampType)

# Explicit schemas: Spark can't infer a type for a column that is empty in every row
# (e.g. weight_min when a guide shows no weights).
EXAMS_SCHEMA = StructType([
    StructField("exam_code", StringType(), False),
    StructField("title", StringType(), True),
    StructField("study_guide_url", StringType(), True),
    StructField("domain_count", LongType(), True),
    StructField("skill_count", LongType(), True),
    StructField("last_synced_utc", TimestampType(), True),
])
SKILLS_SCHEMA = StructType([
    StructField("skill_key", StringType(), False),
    StructField("exam", StringType(), False),
    StructField("domain_number", LongType(), True),
    StructField("domain", StringType(), True),
    StructField("domain_name", StringType(), True),
    StructField("weight_min", LongType(), True),
    StructField("weight_max", LongType(), True),
    StructField("subcategory_number", LongType(), True),
    StructField("subcategory", StringType(), True),
    StructField("skill_number", LongType(), True),
    StructField("skill", StringType(), True),
    StructField("is_current", BooleanType(), True),
    StructField("last_seen_utc", TimestampType(), True),
])


def to_df(rows, schema):
    return spark.createDataFrame([tuple(r.get(f.name) for f in schema.fields) for r in rows], schema)

EXAMS = ["DP-600", "DP-700", "DP-800", "PL-300"]   # add an exam code here to support it

now = datetime.now(timezone.utc)
exam_rows, skill_rows, failures = [], [], []

for code in EXAMS:
    try:
        html = fetch_study_guide(code)
        skills = parse_study_guide(html, code)
        for s in skills:
            s.update(is_current=True, last_seen_utc=now)
        skill_rows += skills
        exam_rows.append({
            "exam_code": code,
            "title": parse_exam_title(html, code),
            "study_guide_url": BASE_URL.format(code=code.lower()),
            "domain_count": len({s["domain_number"] for s in skills}),
            "skill_count": len(skills),
            "last_synced_utc": now,
        })
        print(f"{code}: {len(skills)} skills")
    except Exception as e:  # one broken page shouldn't block the other exams
        failures.append(code)
        print(f"{code}: FAILED - {e}")

synced = [r["exam_code"] for r in exam_rows]

# ---- exams: upsert synced exams, leave failed ones as they were ----------
exams_df = to_df(exam_rows, EXAMS_SCHEMA)
if spark.catalog.tableExists("exams"):
    (DeltaTable.forName(spark, "exams").alias("t")
        .merge(exams_df.alias("s"), "t.exam_code = s.exam_code")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    exams_df.write.format("delta").saveAsTable("exams")

# ---- exam_skills: upsert by skill_key, retire keys no longer on the page -
skills_df = to_df(skill_rows, SKILLS_SCHEMA)
if spark.catalog.tableExists("exam_skills"):
    tbl = DeltaTable.forName(spark, "exam_skills")
    (tbl.alias("t")
        .merge(skills_df.alias("s"), "t.skill_key = s.skill_key")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    current_keys = [r["skill_key"] for r in skill_rows]
    tbl.update(
        condition=F.col("exam").isin(synced) & ~F.col("skill_key").isin(current_keys),
        set={"is_current": F.lit(False)},
    )
else:
    skills_df.write.format("delta").saveAsTable("exam_skills")

if failures:
    raise RuntimeError(f"Study guide sync failed for: {', '.join(failures)}")
